# Challenge 9 — Análisis de Actividad de Producto

## 0. Configuración Inicial

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Cargar el dataset
df = pd.read_csv('docs/product_activity.csv')

# Guardar conteo original antes de cualquier limpieza
RAW_COUNT = len(df)
print(f"Dataset cargado: {RAW_COUNT} filas, {df.shape[1]} columnas")

## 1. Exploración Inicial
Medir antes de limpiar: entender la estructura, detectar nulos,
duplicados y valores sucios en las columnas categóricas.

### 1.1 — Vista general del dataset

In [ ]:
# Primeras filas, tipos de dato y estadísticas numéricas
print("=== head() ===")
display(df.head())

print("\n=== info() ===")
display(df.info())

print("\n=== describe() ===")
display(df.describe())

### 1.2 — Nulos y duplicados exactos

In [ ]:
# Nulos por columna
print("Nulos por columna:")
display(df.isnull().sum())

# Duplicados exactos
dup_count = df.duplicated().sum()
print(f"\nFilas duplicadas exactas: {dup_count}")

### 1.3 — Valores únicos y frecuencias
Detectar variantes sucias (typos, mayúsculas, espacios, valores fuera de diccionario).

In [ ]:
# Revisamos las 3 columnas categóricas clave
for col in ["plan_type", "post_category", "device_type"]:
    print(f"\n{col}:")
    display(df[col].value_counts())

### 1.4 — Chequeos lógicos
¿Hay posts creados antes del signup? ¿`days_since_signup` es consistente?

In [ ]:
# Convertir fechas temporalmente para los chequeos
signup_tmp = pd.to_datetime(df["created_at"], errors="coerce")
post_tmp = pd.to_datetime(df["post_created_at"], errors="coerce")

# Posts que ocurren ANTES del signup
before_signup = (post_tmp < signup_tmp).sum()
print(f"Posts antes del signup: {before_signup}")

# Mismatches en days_since_signup vs cálculo real
mismatch_pre = ((post_tmp - signup_tmp).dt.days != df["days_since_signup"]).sum()
print(f"Mismatches en days_since_signup: {mismatch_pre}")

## 2. Limpieza Básica con Criterio
Eliminar duplicados, normalizar categóricas, convertir fechas,
recalcular `days_since_signup` y separar filas con errores duros.

### 2.1 — Eliminar duplicados exactos

In [ ]:
df = df.drop_duplicates()
print(f"Filas después de eliminar {dup_count} duplicados: {len(df)}")

### 2.2 — Normalización canónica
Mapear variantes a los valores canónicos definidos en el challenge.

In [ ]:
# Paso 1: minúsculas y sin espacios en blanco
for col in ["plan_type", "device_type", "post_category"]:
    df[col] = df[col].str.strip().str.lower()

# Paso 2: corregir typos conocidos en post_category
CATEGORY_MAP = {
    "sport": "sports", "sporst": "sports", "sp0rts": "sports",
    "tehc": "tech",
    "lfe": "life",
    "gamming": "gaming",
    "musc": "music",
    "educatoin": "education",
    "healt": "health",
    "sciense": "science",
    "trvael": "travel",
    "finanse": "finance",
}
df["post_category"] = df["post_category"].replace(CATEGORY_MAP)

# Verificar resultado
for col in ["plan_type", "device_type", "post_category"]:
    print(f"\n{col} después de normalizar:")
    display(df[col].value_counts())

### 2.3 — Conversión de fechas y recálculo de `days_since_signup`

In [ ]:
# Convertir a datetime
df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
df["post_created_at"] = pd.to_datetime(df["post_created_at"], errors="coerce")

# Reportar fechas no parseables
print(f"Fechas no parseables - signup: {df['created_at'].isna().sum()}")
print(f"Fechas no parseables - post:   {df['post_created_at'].isna().sum()}")

# Recálculo obligatorio
df["days_since_signup_calc"] = (df["post_created_at"] - df["created_at"]).dt.days

# Comparar con la columna original
mismatch_count = (df["days_since_signup"] != df["days_since_signup_calc"]).sum()
print(f"\nMismatches en days_since_signup: {mismatch_count}")

### 2.4 — Quarantine (Cuarentena)
Separar filas con errores duros en un DataFrame aparte con `reason_code`.

In [ ]:
# Diccionarios válidos según el challenge
VALID_PLANS = {"free", "pro", "enterprise"}
VALID_DEVICES = {"web", "mobile", "desktop"}
VALID_CATEGORIES = {"tech", "life", "sports", "science", "finance",
                    "gaming", "music", "health", "education", "travel"}

# Condiciones de error "duro" con su etiqueta
conditions = {
    "post_before_signup": df["post_created_at"] < df["created_at"],
    "unparseable_date":   df["created_at"].isna() | df["post_created_at"].isna(),
    "invalid_plan":       ~df["plan_type"].isin(VALID_PLANS),
    "invalid_device":     ~df["device_type"].isin(VALID_DEVICES),
    "invalid_category":   ~df["post_category"].isin(VALID_CATEGORIES),
}

# Construir reason_code concatenando motivos
df["reason_code"] = ""
for reason, mask in conditions.items():
    df.loc[mask, "reason_code"] += reason + ";"

# Separar cuarentena vs core
is_quarantine = df["reason_code"] != ""
df_quarantine = df[is_quarantine].copy()
df_clean = df[~is_quarantine].drop(columns=["reason_code"]).copy()

print(f"Filas en cuarentena: {len(df_quarantine)}")
print(f"Filas en core (limpias): {len(df_clean)}")

## 3. Data Quality Report
Resumen comparando el dataset RAW original contra el dataset CORE limpio.

In [ ]:
quarantine_pct = len(df_quarantine) / RAW_COUNT * 100
mismatch_pct = mismatch_count / RAW_COUNT * 100

report = pd.DataFrame({
    "Métrica": [
        "Filas RAW (originales)",
        "Duplicados removidos",
        "Filas después de dedup",
        "Filas CORE (limpias)",
        "Filas Quarantine",
        "% Quarantine",
        "% Mismatches en fechas",
    ],
    "Valor": [
        RAW_COUNT,
        dup_count,
        len(df),
        len(df_clean),
        len(df_quarantine),
        f"{quarantine_pct:.2f}%",
        f"{mismatch_pct:.2f}%",
    ],
})

display(report)

## 4. Métricas y Análisis
A partir del dataset CORE, calculamos distribuciones de volumen,
engagement por segmento, y diferencias entre nivel evento vs usuario.

### 4.1 — Distribuciones (Volumen)
Usuarios únicos por plan, y actividad (#posts) por país, categoría y dispositivo.

In [ ]:
# Usuarios únicos por plan
print("Usuarios únicos por plan:")
display(df_clean.groupby("plan_type")["user_id"].nunique())

# Actividad (#posts) por país, categoría y dispositivo
for col in ["country", "post_category", "device_type"]:
    print(f"\nActividad por {col}:")
    display(df_clean[col].value_counts())

# Gráfico de barras: posts por plan
df_clean["plan_type"].value_counts().plot(kind="bar", title="Posts por Plan")
plt.ylabel("Cantidad de posts")
plt.tight_layout()
plt.show()

### 4.2 — Engagement (Votos)
Media y mediana de votos por plan, país, categoría y dispositivo.

In [ ]:
# Votos (media y mediana) segmentados por cada dimensión
for col in ["plan_type", "country", "post_category", "device_type"]:
    print(f"\nVotos por {col}:")
    display(df_clean.groupby(col)["votes_received"].agg(["mean", "median"]))

### 4.3 — Promedios e Interpretación

**Interpretación:** La unidad de análisis es el *evento* (cada fila = un post).
Un usuario con muchos posts impacta fuertemente en el promedio general,
ya que sus votos se contabilizan repetidamente.
Esto genera un sesgo hacia los heavy users / outliers.

In [ ]:
# Promedio de votos por plan
print("Promedio de votos por plan:")
display(df_clean.groupby("plan_type")["votes_received"].mean())

# Posts promedio por usuario
avg_posts = df_clean.groupby("user_id")["post_id"].count().mean()
print(f"\nPosts promedio por usuario: {avg_posts:.2f}")

### 4.4 — Evento vs Usuario

**¿Por qué difieren?** A nivel *evento*, los usuarios más activos
sobrerrepresentan la muestra (si un usuario hace 100 posts, su media
pesará 100 veces más). Al calcular por *usuario*, cada persona pesa
igual (1 promedio), eliminando el efecto de los heavy users.

In [ ]:
# Promedio de votos a nivel EVENTO (por fila)
avg_event = df_clean["votes_received"].mean()
print(f"Promedio de votos por evento (fila): {avg_event:.4f}")

# Promedio de votos a nivel USUARIO (agrupado)
avg_user = df_clean.groupby("user_id")["votes_received"].mean().mean()
print(f"Promedio de votos por usuario:       {avg_user:.4f}")

## 5. Concentración y Temporalidad

### 5.1 — Concentración (Top 1%)
¿Qué porcentaje de posts/votos viene del top 1% de usuarios?

In [ ]:
# Posts y votos por usuario
user_posts = df_clean.groupby("user_id")["post_id"].count()
user_votes = df_clean.groupby("user_id")["votes_received"].sum()

# Top 1% de usuarios
top1_posts = user_posts[user_posts >= user_posts.quantile(0.99)]
top1_votes = user_votes[user_votes >= user_votes.quantile(0.99)]

top1_post_pct = top1_posts.sum() / user_posts.sum() * 100
top1_vote_pct = top1_votes.sum() / user_votes.sum() * 100

print(f"Top 1% usuarios por posts: {len(top1_posts)} usuarios")
print(f"Top 1% → {top1_post_pct:.2f}% de los posts")
print(f"Top 1% → {top1_vote_pct:.2f}% de los votos")

### 5.2 — Tendencia temporal
Actividad y engagement por mes.

In [ ]:
# Agrupar por mes
df_clean["month"] = df_clean["post_created_at"].dt.to_period("M")

# Actividad por mes
df_clean.groupby("month")["post_id"].count().plot(
    kind="line", title="Actividad por Mes")
plt.ylabel("Cantidad de posts")
plt.tight_layout()
plt.show()

# Engagement por mes
df_clean.groupby("month")["votes_received"].mean().plot(
    kind="line", title="Engagement por Mes")
plt.ylabel("Votos promedio")
plt.tight_layout()
plt.show()

---
## 6. Product Decisions (Basadas en evidencia)

### 6.1 — Preguntas

**¿Qué segmento priorizarías y por qué?**
El segmento Enterprise/Pro, ya que muestran métricas de engagement consistentemente superiores (mayor promedio de votos por post). Además, enfoques de retención hacia usuarios de mobile suelen ser clave para el crecimiento del producto.

**¿Qué parte del tablero "mentía" antes de limpiar?**
La métrica `days_since_signup`. Mostraba un porcentaje importante de mismatches comparado con un recálculo limpio a partir de las fechas reales, generando posibles errores en el cálculo del tiempo de vida (LTV) del usuario.

**¿Qué nuevo dato agregarías al tracking?**
Tiempo promedio leyendo el post o si incluye multimedia, para que el número de votos no sea el único reflejo de valor real (engagement puro).

### 6.2 — Acciones concretas

**Acción 1 — Campaña de retención:**
El Top 1% de usuarios concentra un porcentaje significativo de posts y votos. Se debe incentivar al resto de la base a publicar, por ejemplo, ofreciendo un "logro/bonus" en el primer mes de uso. *Respaldo: métricas de concentración del Top 1% en sección 5.1.*

**Acción 2 — Mejora de experiencia mobile:**
Si la categoría o dispositivo móvil domina en volumen de actividad, se debe priorizar recursos de ingeniería en optimizar su interfaz en próximas versiones. *Respaldo: distribución de actividad por dispositivo en sección 4.1.*

### Limitaciones del dataset
- No hay datos de retención ni duración de sesión, limitando la evaluación de engagement real.
- `user_total_posts` es un atributo repetido por fila, lo cual puede sesgar promedios si no se agrupa por usuario.
- No hay información sobre la calidad del contenido del post (longitud, multimedia, etc.).
- El período temporal del dataset puede no ser representativo de tendencias a largo plazo.
- La columna `days_since_signup` original no es confiable; el análisis usa la versión recalculada.

## 7. Exportación de Archivos

In [ ]:
# 1. Dataset limpio (core)
df_clean.to_csv("clean_product_activity.csv", index=False)
print("Exportado: clean_product_activity.csv")

# 2. Dataset cuarentena (con motivo de exclusión)
df_quarantine.to_csv("quarantine_product_activity.csv", index=False)
print("Exportado: quarantine_product_activity.csv")

# 3. Tabla resumen de métricas clave
metrics = pd.DataFrame({
    "Métrica": ["unique_users", "total_core_posts",
                "avg_votes_per_event", "avg_votes_per_user",
                "top1_post_pct", "top1_vote_pct"],
    "Valor": [df_clean["user_id"].nunique(), len(df_clean),
              avg_event, avg_user,
              top1_post_pct, top1_vote_pct],
})
metrics.to_csv("metrics_summary.csv", index=False)
print("Exportado: metrics_summary.csv")

print("\nExportación completa.")